# case39 model, severe load loss, and deterministic baseline

This notebook is part of the reproducible case39 study for the Suboptimal Control paper. Heavy Monte Carlo results used in the manuscript are cached in `../results/`; the reusable simulation and certificate code is in `../src/`.

In [ ]:
from pathlib import Path
import json, csv, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('..').resolve()
RES=ROOT/'results'; FIG=ROOT/'figures'
print(ROOT)

## Model reduction

We use the same lossless fixed-voltage reduction as the companion cycle-space paper.  With incidence matrix $B$, line strengths $K_e$, post-event injection vector $p^+$ and direct generator adjustment $S_g u$, the stochastic swing equations are
$$d\theta=\omega\,dt,$$
$$M d\omega=[p^+-C\omega-BK\sin(B^T\theta)+S_g u]dt+\Gamma dW.$$
The severe event is the disappearance of the load at MATPOWER bus 20 at the companion-paper loading factor $\alpha=4$. The cached metadata records the exact load loss and the deterministic regression values.

In [ ]:
meta=json.loads((RES/'metadata.json').read_text()); meta

## Deterministic transient regression

The common post-event equilibrium uses equal generator balancing.  Before adding stochasticity or feedback we reproduce the deterministic transient.  The baseline is deliberately close to the cohesion boundary: the maximum energized-line angle rises from about $46.17^\circ$ pre-event to about $81.11^\circ$.

In [ ]:
df=pd.read_csv(RES/'deterministic_transient.csv');
fig,ax=plt.subplots(1,2,figsize=(9,3)); ax[0].plot(df.time,df.max_edge_angle_deg); ax[0].axhline(90,ls='--'); ax[1].plot(df.time,df.max_abs_frequency); plt.show()

## Static balancing concentration

The companion cycle-space study parameterizes how concentrated the balancing is among generators.  Here we keep that parameter as a diagnostic: concentration quickly pushes the deterministic and stochastic transients through the cohesion boundary.  This is why the equal-generator final allocation is used as the common target in the H0--H3 transient-control comparison.

In [ ]:
b=json.loads((RES/'balance_concentration.json').read_text()); pd.DataFrame(b)

## Recomputing from MATPOWER

For a full rerun, place standard MATPOWER `case39.m` in `../cyclespace_base/data/` or allow the loader to download it.  The scripts in `../cyclespace_base/` reproduce the lossless model and the deterministic companion results.  The stochastic extension imports those objects through `../src/stochastic_swing.py`.